In [1]:
!pip install cvxpy control --quiet
print('OK')

OK



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install casadi


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
"""
KoopmanMPC v5 — PARÁMETROS PLANTA REAL
=======================================
Actualizado desde la versión benchmark (TU Wien) a los parámetros
calibrados experimentalmente de la planta física construida.

CAMBIOS respecto a versión benchmark:
  [P1] Parámetros físicos → planta real calibrada
  [P2] Referencias: 0.10→0.25→0.15 → 0.10→0.20→0.15 m
  [P3] Rango de estados: clip a 0.25 m (altura física de los tanques)
  [P4] Datos EDMD: rango operativo [0.05, 0.22] m
  [P5] σ = 1e-3 m (HC-SR04) — solo para reporte, el MPC usa estado real

Mantiene sin cambio:
  - Diccionario 8 observables: [√h1,√h2,√h3, h1h2,h2h3,h1h3, h1,h3]
  - N=20, q=500, R1=diag(0.1,1.0), R2=diag(0.1,0.1)
  - Solver sqpmethod + qrqp con shifted warm-start
  - Exclusión de transitorios (60s) en RMSE_SS
"""

import numpy as np
from scipy.optimize import least_squares
import casadi as ca
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import time

warnings.filterwarnings('ignore')
output_dir = Path('./outputs_sim')
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# [P1] PARÁMETROS FÍSICOS — PLANTA REAL CALIBRADA
# ============================================================================
p = {
    'Atank'   : 3.02e-3,
    'rho'     : 997.0,   'eta'      : 8.9e-4,   'g'        : 9.81,
    'alphao1' : 0.0271,  'Do1'      : 10e-3,
    'alphao2' : 0.0271,  'A2'       : 7.85e-5,
    'alphao3' : 0.0271,  'Do3'      : 10e-3,
    'alpha120': 0.3038,  'D12'      : 10e-3,
    'A12'     : 7.85e-5, 'lambdac12': 24000,
    'alpha230': 0.1344,  'D23'      : 10e-3,
    'A23'     : 7.85e-5, 'lambdac23': 29600,
    'qi1max'  : 2.69e-5,
    'qi3max'  : 2.41e-5,
}

Ts = 2.0; T_sim = 800; Nsim = int(T_sim / Ts); EPS = 1e-10; EPS_PSI = 1e-8

# ============================================================================
# MODELO
# ============================================================================
def nl_ode(x, u, params=None):
    if params is None: params = p
    h1 = max(x[0], EPS); h2 = max(x[1], EPS); h3 = max(x[2], EPS)
    qi1 = params['qi1max'] * np.clip(u[0], 0, 1)
    qi3 = params['qi3max'] * np.clip(u[1], 0, 1)
    qo1 = params['alphao1'] * (params['Do1']**2 * np.pi / 4) * np.sqrt(2 * params['g'] * h1)
    qo2 = params['alphao2'] * params['A2']                    * np.sqrt(2 * params['g'] * h2)
    qo3 = params['alphao3'] * (params['Do3']**2 * np.pi / 4) * np.sqrt(2 * params['g'] * h3)
    dh12 = h1 - h2
    l12  = params['D12'] * params['rho'] / params['eta'] * np.sqrt(2 * params['g'] * abs(dh12) + EPS)
    a12  = params['alpha120'] * np.tanh(2 * l12 / params['lambdac12'])
    q12  = a12 * params['A12'] * np.sqrt(2 * params['g'] * abs(dh12) + EPS) * np.sign(dh12 + 1e-15)
    dh23 = h2 - h3
    l23  = params['D23'] * params['rho'] / params['eta'] * np.sqrt(2 * params['g'] * abs(dh23) + EPS)
    a23  = params['alpha230'] * np.tanh(2 * l23 / params['lambdac23'])
    q23  = a23 * params['A23'] * np.sqrt(2 * params['g'] * abs(dh23) + EPS) * np.sign(dh23 + 1e-15)
    return np.array([
        (qi1 - q12 - qo1) / params['Atank'],
        (q12 - q23 - qo2) / params['Atank'],
        (qi3 + q23 - qo3) / params['Atank'],
    ])

def rk4_step(x, u, dt=Ts, params=None):
    k1 = nl_ode(x, u, params); k2 = nl_ode(x + dt/2 * k1, u, params)
    k3 = nl_ode(x + dt/2 * k2, u, params); k4 = nl_ode(x + dt * k3, u, params)
    return np.clip(x + dt/6 * (k1 + 2*k2 + 2*k3 + k4), 0, 0.25)  # [P3]

def psi(x):
    h1, h2, h3 = max(x[0], EPS_PSI), max(x[1], EPS_PSI), max(x[2], EPS_PSI)
    return np.array([
        np.sqrt(h1), np.sqrt(h2), np.sqrt(h3),
        h1*h2, h2*h3, h1*h3,
        h1, h3,
    ], dtype=float)

def compute_ss(h2_ref, params=None):
    if params is None: params = p
    def eqs(v):
        h1, h3, u1 = v
        return nl_ode([max(h1, 1e-4), h2_ref, max(h3, 1e-4)],
                      [np.clip(u1, 0, 1), 0.0], params)
    guesses = [
        (h2_ref*1.4, h2_ref*0.8, 0.5), (h2_ref*1.6, h2_ref*0.7, 0.7),
        (h2_ref*1.2, h2_ref*0.9, 0.4), (h2_ref*2.0, h2_ref*0.8, 0.9),
        (h2_ref*2.5, h2_ref*0.8, 0.7), (h2_ref*2.5, h2_ref*0.8, 0.75),
        (h2_ref*1.8, h2_ref*0.85, 0.85),
    ]
    best, best_res = None, 1e10
    for g in guesses:
        try:
            s = least_squares(eqs, list(g),
                              bounds=([0.01, 0.01, 0.0], [0.25, 0.25, 1.0]),
                              max_nfev=5000)
            if s.success and np.linalg.norm(s.fun) < best_res:
                best_res = np.linalg.norm(s.fun); best = s.x
        except: pass
    if best is not None and best_res < 1e-4:
        return np.array([best[0], h2_ref, best[1]]), np.array([np.clip(best[2], 0, 1), 0.0])
    return None, None

print("=" * 70)
print("  KoopmanMPC v5 — Simulación Parámetros Planta Real")
print("  Diccionario: [√h₁,√h₂,√h₃, h₁h₂,h₂h₃,h₁h₃, h₁,h₃]")
print("  Ref: 0.10 → 0.20 → 0.15 m")
print("=" * 70)

print("\nCalculando estados estacionarios...")
SS = {}
for h2r in [0.10, 0.15, 0.20]:
    x_ss, u_ss = compute_ss(h2r)
    if x_ss is not None:
        SS[h2r] = (x_ss, u_ss)
        print(f"  h2={h2r:.2f}m → x_ss={np.round(x_ss, 4)}, u_ss={np.round(u_ss, 4)} ✓")
    else:
        print(f"  h2={h2r:.2f}m → SS NO encontrado ✗")

# ============================================================================
# DATOS EDMD
# ============================================================================
print("\nEntrenando EDMD...")

def collect_edmd_data(seed=42):
    rng = np.random.default_rng(seed)
    z_k, u_k, z_k1 = [], [], []

    for h2r in [0.08, 0.10, 0.12, 0.15, 0.17, 0.20, 0.22]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None: continue
        perts = [0.95, 0.90, 0.98, 0.85, 0.92, 0.88, 1.00, 0.80, 1.05, 0.75, 0.97, 0.83]
        for pert in perts:
            x = x_ss.copy(); x[1] = np.clip(h2r * pert, 0.04, 0.22)
            for _ in range(180):
                err = x[1] - h2r; u = u_ss.copy()
                u[0] += np.clip(-0.4 * err, -0.4, 0.4) + rng.uniform(-0.12, 0.12)
                u[1] += rng.uniform(-0.04, 0.04); u = np.clip(u, 0, 1)
                z = psi(x); xn = rk4_step(x, u, Ts); zn = psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None: continue
        for pert in [0.65, 0.70, 0.75, 1.20, 1.25, 1.30]:
            x = x_ss.copy(); x[1] = np.clip(h2r * pert, 0.04, 0.22)
            for _ in range(100):
                err = x[1] - h2r; u = u_ss.copy()
                u[0] += np.clip(-0.6 * err, -0.6, 0.6) + rng.uniform(-0.1, 0.1)
                u = np.clip(u, 0, 1)
                z = psi(x); xn = rk4_step(x, u, Ts); zn = psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    for (h2_from, h2_to) in [(0.10, 0.20), (0.20, 0.15), (0.15, 0.10)]:
        xf, uf = compute_ss(h2_from)
        xt, _  = compute_ss(h2_to)
        if xf is None or xt is None: continue
        x = xf.copy()
        for _ in range(200):
            err = x[1] - h2_to; u = uf.copy()
            u[0] += np.clip(-0.5 * err, -0.5, 0.5) + rng.uniform(-0.08, 0.08)
            u = np.clip(u, 0, 1)
            z = psi(x); xn = rk4_step(x, u, Ts); zn = psi(xn)
            z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    return np.array(z_k), np.array(u_k), np.array(z_k1)

Z, U, Zn = collect_edmd_data(seed=42)
nz, nu   = Z.shape[1], U.shape[1]
print(f"  {len(Z)} muestras recolectadas, nz={nz}")

Phi   = np.hstack([Z, U]); lam = 1e-4
Theta = np.linalg.solve(Phi.T @ Phi + lam * np.eye(nz + nu), Phi.T @ Zn)
A_raw = Theta[:nz, :].T; B_d = Theta[nz:, :].T
vals, vecs = np.linalg.eig(A_raw); delta = 1e-4
vals_s = np.where(np.abs(vals) >= 1.0, vals / np.abs(vals) * (1.0 - delta), vals)
A_d    = np.real(vecs @ np.diag(vals_s) @ np.linalg.inv(vecs))
eigs   = np.linalg.eigvals(A_d)
rmse_id = np.sqrt(np.mean((Z @ A_d.T + U @ B_d.T - Zn)**2))
print(f"  RMSE one-step: {rmse_id:.2e}")
print(f"  |λ| ∈ [{np.abs(eigs).min():.5f}, {np.abs(eigs).max():.5f}] Estable ✓")

# ============================================================================
# CONSTRUCCIÓN DEL SOLVER MPC
# ============================================================================
print("\nConstruyendo KoopmanMPC...")
N_mpc = 20; q_mpc = 500; wT = 10
R1 = np.diag([0.1, 1.0]); R2 = np.diag([0.1, 0.1])
umin = np.array([0.0, 0.0]); umax = np.array([1.0, 1.0])

Z_sym  = ca.SX.sym('Z', nz, N_mpc+1); U_sym = ca.SX.sym('U', nu, N_mpc)
z0_sym = ca.SX.sym('z0', nz); yr_sym = ca.SX.sym('yr'); up_sym = ca.SX.sym('up', nu)
A_cas  = ca.DM(A_d); B_cas = ca.DM(B_d)
J = 0; g = []; lbg = []; ubg = []
g += [Z_sym[:, 0] - z0_sym]; lbg += [0.0]*nz; ubg += [0.0]*nz
for k in range(N_mpc):
    z_next = A_cas @ Z_sym[:, k] + B_cas @ U_sym[:, k]
    g += [Z_sym[:, k+1] - z_next]; lbg += [0.0]*nz; ubg += [0.0]*nz
    h2_k = Z_sym[1, k+1]**2; ey = h2_k - yr_sym
    du   = U_sym[:, 0] - up_sym if k == 0 else U_sym[:, k] - U_sym[:, k-1]
    wk   = wT if k == N_mpc - 1 else 1
    J += wk * q_mpc * ey**2
    J += ca.mtimes(U_sym[:, k].T, ca.DM(R1) @ U_sym[:, k])
    J += ca.mtimes(du.T, ca.DM(R2) @ du)

w_sym   = ca.vertcat(ca.reshape(Z_sym, nz*(N_mpc+1), 1), ca.reshape(U_sym, nu*N_mpc, 1))
par_sym = ca.vertcat(z0_sym, yr_sym, up_sym)
lbw = ([-ca.inf] * (nz*(N_mpc+1)) + list(np.tile(umin, N_mpc)))
ubw = ([ ca.inf] * (nz*(N_mpc+1)) + list(np.tile(umax, N_mpc)))
nlp  = {'x': w_sym, 'f': J, 'g': ca.vertcat(*g), 'p': par_sym}
opts = {'qpsol': 'qrqp',
        'qpsol_options': {'print_iter': False, 'print_header': False},
        'print_iteration': False, 'print_header': False,
        'print_status': False, 'max_iter': 200,
        'tol_pr': 1e-4, 'tol_du': 1e-4, 'verbose': False, 'print_time': False}
solver_kmpc = ca.nlpsol('solver_kmpc', 'sqpmethod', nlp, opts)
print("  KoopmanMPC ✓")

def warm_start_koopman(z0, uprev):
    Z0 = np.zeros((N_mpc+1, nz)); Z0[0] = z0
    for i in range(N_mpc):
        Z0[i+1] = A_d @ Z0[i] + B_d @ uprev
        Z0[i+1, :3] = np.maximum(Z0[i+1, :3], 0)
    return np.concatenate([Z0.flatten(), np.tile(uprev, (N_mpc, 1)).flatten()])

def koopman_mpc_step(x_k, yr, uprev, w0):
    z0 = psi(x_k)
    if w0 is None:
        w0 = warm_start_koopman(z0, uprev)
    par_val = np.concatenate([z0, [yr], uprev])
    sol  = solver_kmpc(x0=w0, lbx=lbw, ubx=ubw, lbg=lbg, ubg=ubg, p=par_val)
    w_opt = np.array(sol['x']).flatten()
    Z_opt = w_opt[:nz*(N_mpc+1)].reshape(N_mpc+1, nz)
    U_opt = w_opt[nz*(N_mpc+1):].reshape(N_mpc, nu)
    u_k   = np.clip(U_opt[0], umin, umax)
    Z_shift = np.vstack([Z_opt[1:], Z_opt[-1:]])
    U_shift = np.vstack([U_opt[1:], U_opt[-1:]])
    return u_k, np.concatenate([Z_shift.flatten(), U_shift.flatten()])

# ============================================================================
# SIMULACIÓN — 0.10 → 0.20 → 0.15 m [P2]
# ============================================================================
print("\n" + "=" * 70)
print("  SIMULACIÓN: 0.10 → 0.20 → 0.15 m | T=800s")
print("=" * 70)

t1 = int(Nsim * 0.33); t2 = int(Nsim * 0.66)
yref_v = np.concatenate([
    0.10 * np.ones(t1),
    0.20 * np.ones(t2 - t1),
    0.15 * np.ones(Nsim - t2)
])
ref_change_times_s = [t1 * Ts, t2 * Ts]

x_ss_init, u_ss_init = SS[0.10]
x0_true = x_ss_init.copy()
print(f"  Condición inicial (SS h2=0.10m): {np.round(x0_true, 4)}")

T_hist = np.arange(Nsim + 1) * Ts
X_hist = np.zeros((3, Nsim+1)); U_hist = np.zeros((2, Nsim)); step_ms = []
X_hist[:, 0] = x0_true
x = x0_true.copy(); uprev = u_ss_init.copy(); w0 = None
t_start = time.time()

for k in range(Nsim):
    yr = yref_v[k]
    t0 = time.time()
    u_k, w0 = koopman_mpc_step(x, yr, uprev, w0)
    step_ms.append((time.time() - t0) * 1000)
    U_hist[:, k] = u_k; uprev = u_k.copy()
    x = rk4_step(x, u_k, Ts); X_hist[:, k+1] = x
    if k % 50 == 0:
        print(f"  t={k*Ts:5.0f}s | h2={x[1]:.4f}m | ref={yr:.4f}m | "
              f"err={abs(x[1]-yr)*100:.2f}cm | {step_ms[-1]:.1f}ms")

total_s = time.time() - t_start

# ============================================================================
# MÉTRICAS
# ============================================================================
SETTLING_S = 60
ref_full = np.append(yref_v, yref_v[-1])
e_h2_full = (X_hist[1, :] - ref_full) * 100
mask_400 = T_hist > 400
settling_mask = np.ones(len(T_hist), dtype=bool)
for tc in ref_change_times_s:
    settling_mask &= ~((T_hist >= tc) & (T_hist < tc + SETTLING_S))
mask_ss = mask_400 & settling_mask

rmse_400 = np.sqrt(np.mean(e_h2_full[mask_400]**2))
rmse_ss  = np.sqrt(np.mean(e_h2_full[mask_ss]**2))
avg_ms   = np.mean(step_ms); p99_ms = np.percentile(step_ms, 99)

print("\n" + "=" * 70)
print("  MÉTRICAS KoopmanMPC — Simulación Planta Real")
print("=" * 70)
print(f"  RMSE h2 (t>400s, incl. transitorios):     {rmse_400:.4f} cm")
print(f"  RMSE h2 (estado estacionario, SS):         {rmse_ss:.4f} cm  ← métrica comparable")
print(f"  Tiempo total simulación:                   {total_s:.1f} s")
print(f"  Tiempo medio/paso:                         {avg_ms:.1f} ms/paso")
print(f"  Tiempo p99/paso:                           {p99_ms:.1f} ms/paso")

# ============================================================================
# GRÁFICAS — estilo unificado con Koopman Closed-Loop v3
# ============================================================================
fig, axes = plt.subplots(3, 2, figsize=(13, 11))
fig.suptitle(
    f'KoopmanMPC — Simulación\n'
    f'RMSE_SS={rmse_ss:.3f} cm | media={avg_ms:.1f} ms/paso | p99={p99_ms:.1f} ms/paso',
    fontsize=9, fontweight='bold')

# ── Panel 1: h1 ─────────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(T_hist, X_hist[0]*100, 'b', lw=1.5, label='$h_1$ real')
ax.set_ylabel('$h_1$ [cm]'); ax.set_xlabel('t [s]')
ax.set_title('Tanque 1 (estado completo)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 2: h2 con referencia ──────────────────────────────────────────────
ax = axes[0, 1]
ax.plot(T_hist, X_hist[1]*100, 'b', lw=2, label='$h_2$ real')
ax.stairs(ref_full[:-1]*100, T_hist, color='k', linestyle='--', lw=1.5, label='Referencia')
ax.set_ylabel('$h_2$ [cm]'); ax.set_xlabel('t [s]')
ax.set_title('Tanque 2 — salida controlada')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 3: h3 ─────────────────────────────────────────────────────────────
ax = axes[1, 0]
ax.plot(T_hist, X_hist[2]*100, color=[0, .6, 0], lw=1.5, label='$h_3$ real')
ax.set_ylabel('$h_3$ [cm]'); ax.set_xlabel('t [s]')
ax.set_title('Tanque 3 (estado completo)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 4: error de seguimiento ───────────────────────────────────────────
ax = axes[1, 1]
ax.plot(T_hist, e_h2_full, 'r', lw=1.5,
        label=f'RMSE_SS={rmse_ss:.3f} cm')
ax.axhline(0, color='k', ls=':')
ax.axvline(400, color='gray', lw=0.8, ls=':', label='t=400s')
for i, tc in enumerate(ref_change_times_s):
    ax.axvspan(tc, tc + SETTLING_S, alpha=0.12, color='orange',
               label='transitorio excluido' if i == 0 else None)
ax.set_ylabel('Error $h_2$ [cm]'); ax.set_xlabel('t [s]')
ax.set_title('Error de seguimiento')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 5: señales de control ─────────────────────────────────────────────
ax = axes[2, 0]
ax.stairs(U_hist[0], T_hist, color='b', lw=1.5, label='$u_1$')
ax.stairs(U_hist[1], T_hist, color='r', lw=1.5, label='$u_3$')
ax.set_ylim([-0.05, 1.1]); ax.set_ylabel('Control [0–1]'); ax.set_xlabel('t [s]')
ax.set_title('Señales de control')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ── Panel 6: tiempo de cómputo ──────────────────────────────────────────────
ax = axes[2, 1]
ax.plot(step_ms, 'k', lw=0.8, alpha=0.7)
ax.axhline(avg_ms, color='b', lw=1.5, ls='--', label=f'Media={avg_ms:.1f} ms')
ax.axhline(p99_ms, color='r', lw=1.5, ls=':',  label=f'P99={p99_ms:.1f} ms')
ax.set_ylabel('Tiempo [ms]'); ax.set_xlabel('Paso k')
ax.set_title('Tiempo de cómputo por paso')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
fname = output_dir / 'koopman_mpc_v5_planta_real.png'
plt.savefig(fname, dpi=200, bbox_inches='tight'); plt.close()
print(f"\nFigura guardada: {fname}")

  KoopmanMPC v5 — Simulación Parámetros Planta Real
  Diccionario: [√h₁,√h₂,√h₃, h₁h₂,h₂h₃,h₁h₃, h₁,h₃]
  Ref: 0.10 → 0.20 → 0.15 m

Calculando estados estacionarios...
  h2=0.10m → x_ss=[0.114  0.1    0.0816], u_ss=[0.3293 0.    ] ✓
  h2=0.15m → x_ss=[0.1676 0.15   0.1267], u_ss=[0.4043 0.    ] ✓
  h2=0.20m → x_ss=[0.2207 0.2    0.1725], u_ss=[0.4664 0.    ] ✓

Entrenando EDMD...
  15360 muestras recolectadas, nz=8
  RMSE one-step: 8.62e-05
  |λ| ∈ [0.06070, 0.99990] Estable ✓

Construyendo KoopmanMPC...
  KoopmanMPC ✓

  SIMULACIÓN: 0.10 → 0.20 → 0.15 m | T=800s
  Condición inicial (SS h2=0.10m): [0.114  0.1    0.0816]
  t=    0s | h2=0.1000m | ref=0.1000m | err=0.00cm | 1.0ms
  t=  100s | h2=0.0998m | ref=0.1000m | err=0.02cm | 1.0ms
  t=  200s | h2=0.0998m | ref=0.1000m | err=0.02cm | 1.0ms
  t=  300s | h2=0.1967m | ref=0.2000m | err=0.33cm | 1.0ms
  t=  400s | h2=0.2000m | ref=0.2000m | err=0.00cm | 1.0ms
  t=  500s | h2=0.2000m | ref=0.2000m | err=0.00cm | 1.0ms
  t=  600s | h2=0